In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from rasterstats import zonal_stats
import rasterio
from shapely.geometry import mapping

FARMS_PATH = "GeoFiles/CSB/CSB_AZ_cleaned.shp"
RASTER_DIR = "GeoFiles/Temperature_800m/Reprojected/"  
OUT_DIR    = "GeoFiles/Temperature_800m/FarmWise/"

YEARS = range(2019, 2025)  # 2019–2024

gdf = gpd.read_file(FARMS_PATH)
print(f"Loaded {len(gdf)} farms")

for year in YEARS:
    raster_path = f"{RASTER_DIR}prism_tmean_us_30s_{year}.tif"
    out_path    = f"{OUT_DIR}Farm_Temperature_{year}.shp"

    print(f"\nProcessing year {year} ...")

    # Read raster nodata
    with rasterio.open(raster_path) as src:
        nodata = src.nodata
        arr = src.read(1)
        transform = src.transform

        # precompute valid pixel coords
        rows, cols = np.where(arr != nodata)
        if len(rows) == 0:
            raise ValueError(f"No valid pixels found in {raster_path}")
        valid_coords = np.array(rasterio.transform.xy(transform, rows, cols)).T
        valid_values = arr[rows, cols]

    # Run zonal stats
    stats = zonal_stats(
        gdf,
        raster_path,
        stats=["mean"],
        nodata=nodata,
        all_touched=True
    )
    df_stats = pd.DataFrame(stats)
    df_stats.index = gdf.index

    # Attach results
    gdf[f"Temp_{year}"] = df_stats["mean"]

    # Rescue NaNs using nearest valid pixel
    nan_idx = gdf[gdf[f"Temp_{year}"].isna()].index
    for idx in nan_idx: # for each farm with NaN value 
        centroid = gdf.loc[idx].geometry.centroid # get centroid
        cx, cy = centroid.x, centroid.y # centroid coords
        d2 = (valid_coords[:,0] - cx)**2 + (valid_coords[:,1] - cy)**2 # squared distances
        nearest_idx = np.argmin(d2) # index of nearest valid pixel
        gdf.at[idx, f"Temp_{year}"] = valid_values[nearest_idx] # assign value
    
    #Categorizing the Temp_year values
    bins = [-1e9, 17.5, 19.5, 21.5, 22.5, 1e9]  # °C
    labels = ["Very Low","Low","Medium","High","Very High"]
    gdf[f"TempC_{year}"] = pd.cut(gdf[f"Temp_{year}"], bins=bins, labels=labels).astype(str)

    nan_after = gdf[f"Temp_{year}"].isna().sum()
    print(f"NaN farms after rescue: {nan_after}")

    gdf = gdf[['CSBID', 'CSBACRES', 'CNTY', 'geometry', f'Temp_{year}', f'TempC_{year}']].copy()
    
    # Save shapefile for this year
    gdf.to_file(out_path, driver="ESRI Shapefile")
    print(f"Saved {out_path}")

print("\nAll done!")



In [ ]:
#print values and counts of TempC_2019 column
print(gdf["TempC_2024"].value_counts(dropna=False))